# ДЗ 2.3. Связанные колебания и жёсткость

**Уровень:** 🟢 (биения)

Две одинаковые массы $m$ соединены пружинами жёсткости $k_0$ с неподвижными опорами.
Между массами — дополнительная пружина жёсткости $k_c$. Демпфирование отсутствует ($c = 0$).

Уравнения движения:

$$
m\ddot x_1 = -k_0 x_1 - k_c(x_1 - x_2), \qquad
m\ddot x_2 = -k_0 x_2 - k_c(x_2 - x_1).
$$

Начальные условия: первая масса смещена, вторая в покое.

**Задачи:**
- Реализовать решение методами RK4 и Верле.
- При слабой связи $k_c \ll k_0$ наблюдать биения.
- Сравнить численные частоты с собственными.
- Повторить расчёт для нескольких шагов.

**Артефакты:**
- Графики смещений обеих масс.
- Графики кинетической энергии каждой массы и полной механической энергии.

## 1. Аналитическое решение

Собственные частоты системы:

$$
\omega_1 = \sqrt{\frac{k_0}{m}}, \qquad
\omega_2 = \sqrt{\frac{k_0 + 2k_c}{m}}.
$$

При начальных условиях $x_1(0) = A$, $x_2(0) = 0$, $\dot x_1(0) = \dot x_2(0) = 0$
точное решение:

$$
x_1(t) = \frac{A}{2}\bigl(\cos\omega_1 t + \cos\omega_2 t\bigr),\qquad
x_2(t) = \frac{A}{2}\bigl(\cos\omega_1 t - \cos\omega_2 t\bigr).
$$

Частота биений: $\omega_{\text{б}} = (\omega_2 - \omega_1)/2$,
период биений: $T_{\text{б}} = 2\pi / \omega_{\text{б}}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Параметры системы
m  = 1.0
k0 = 10.0    # жёсткость пружин к опорам
kc = 0.1     # жёсткость соединительной пружины (слабая связь: kc << k0)

# Начальные условия: смещаем первую массу, вторая в покое
A = 1.0
state0 = np.array([A, 0.0, 0.0, 0.0])  # [x1, v1, x2, v2]

# Собственные частоты
omega1 = np.sqrt(k0 / m)
omega2 = np.sqrt((k0 + 2*kc) / m)
omega_beat = (omega2 - omega1) / 2
T_beat = 2*np.pi / omega_beat

print(f"omega_1 = {omega1:.4f} рад/с  (синфазная мода)")
print(f"omega_2 = {omega2:.4f} рад/с  (противофазная мода)")
print(f"omega_б = {omega_beat:.4f} рад/с")
print(f"T_б     = {T_beat:.2f} с")

## 2. Методы интегрирования: RK4 и Верле

In [2]:
def rhs(state):
    """Правая часть: [x1, v1, x2, v2] -> [v1, a1, v2, a2]."""
    x1, v1, x2, v2 = state
    a1 = (-k0*x1 - kc*(x1 - x2)) / m
    a2 = (-k0*x2 - kc*(x2 - x1)) / m
    return np.array([v1, a1, v2, a2])


def accel(x1, x2):
    """Ускорения по координатам (для Верле)."""
    a1 = (-k0*x1 - kc*(x1 - x2)) / m
    a2 = (-k0*x2 - kc*(x2 - x1)) / m
    return a1, a2


def step_rk4(state, h):
    k1 = rhs(state)
    k2 = rhs(state + h*k1/2)
    k3 = rhs(state + h*k2/2)
    k4 = rhs(state + h*k3)
    return state + h*(k1 + 2*k2 + 2*k3 + k4)/6


def step_verlet(state, h):
    """Скоростной метод Верле."""
    x1, v1, x2, v2 = state
    a1, a2 = accel(x1, x2)
    # Новые координаты
    x1_new = x1 + h*v1 + 0.5*h*h*a1
    x2_new = x2 + h*v2 + 0.5*h*h*a2
    # Новые ускорения
    a1_new, a2_new = accel(x1_new, x2_new)
    # Новые скорости
    v1_new = v1 + 0.5*h*(a1 + a1_new)
    v2_new = v2 + 0.5*h*(a2 + a2_new)
    return np.array([x1_new, v1_new, x2_new, v2_new])


def simulate(step_func, s0, h, t_end):
    """Интегрирование от 0 до t_end с шагом h."""
    n = int(round(t_end / h))
    t = np.arange(n + 1) * h
    states = np.empty((n + 1, 4))
    states[0] = s0
    for i in range(n):
        states[i+1] = step_func(states[i], h)
    return t, states

## 3. Точное решение

In [3]:
def exact_solution(t):
    x1 = A/2 * (np.cos(omega1*t) + np.cos(omega2*t))
    x2 = A/2 * (np.cos(omega1*t) - np.cos(omega2*t))
    v1 = -A/2 * (omega1*np.sin(omega1*t) + omega2*np.sin(omega2*t))
    v2 = -A/2 * (omega1*np.sin(omega1*t) - omega2*np.sin(omega2*t))
    return x1, v1, x2, v2

## 4. Расчёт для нескольких шагов

In [4]:
t_end = 3 * T_beat  # три периода биений
dt_list = [0.05, 0.02, 0.01, 0.005]

results = {}  # ключ: (метод, dt)
for dt in dt_list:
    t_rk4, s_rk4 = simulate(step_rk4, state0, dt, t_end)
    t_ver, s_ver = simulate(step_verlet, state0, dt, t_end)
    results[("RK4", dt)]   = (t_rk4, s_rk4)
    results[("Верле", dt)] = (t_ver, s_ver)

## 5. Графики смещений обеих масс

In [ ]:
fig, axes = plt.subplots(len(dt_list), 2, figsize=(14, 3*len(dt_list)),
                         constrained_layout=True)
fig.suptitle("Смещения двух масс (биения при слабой связи)", fontsize=13)

for row, dt in enumerate(dt_list):
    t = results[("RK4", dt)][0]
    x1_ex, _, x2_ex, _ = exact_solution(t)

    for col, (mass_idx, label) in enumerate([(0, r"$x_1$"), (2, r"$x_2$")]):
        ax = axes[row, col]

        # Точное решение
        ax.plot(t, x1_ex if col == 0 else x2_ex,
                'k-', lw=1.2, alpha=0.4, label='Точное')

        # RK4
        s_rk4 = results[("RK4", dt)][1]
        ax.plot(t, s_rk4[:, mass_idx], '-', color='tab:blue',
                lw=0.9, label='RK4')

        # Верле
        s_ver = results[("Верле", dt)][1]
        ax.plot(t, s_ver[:, mass_idx], '--', color='tab:orange',
                lw=0.9, label='Верле')

        ax.set_title(f"{label}, h = {dt}", fontsize=10)
        ax.set_xlabel("t, с")
        ax.set_ylabel(label)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.show()

## 6. Кинетическая энергия каждой массы и полная механическая энергия

In [6]:
def kinetic_energy(states):
    """Кинетические энергии T1 и T2."""
    v1 = states[:, 1]
    v2 = states[:, 3]
    T1 = 0.5 * m * v1**2
    T2 = 0.5 * m * v2**2
    return T1, T2


def potential_energy(states):
    """Потенциальная энергия всех пружин."""
    x1 = states[:, 0]
    x2 = states[:, 2]
    return 0.5*k0*x1**2 + 0.5*k0*x2**2 + 0.5*kc*(x1 - x2)**2


def total_energy(states):
    T1, T2 = kinetic_energy(states)
    U = potential_energy(states)
    return T1 + T2 + U

In [ ]:
fig, axes = plt.subplots(len(dt_list), 2, figsize=(14, 3*len(dt_list)),
                         constrained_layout=True)
fig.suptitle("Кинетическая энергия масс и полная энергия", fontsize=13)

for row, dt in enumerate(dt_list):
    for col, (method, color) in enumerate([("RK4", "tab:blue"),
                                           ("Верле", "tab:orange")]):
        t, states = results[(method, dt)]
        T1, T2 = kinetic_energy(states)
        E = total_energy(states)

        ax = axes[row, col]
        ax.plot(t, T1, '-',  lw=0.8, label=r'$T_1$')
        ax.plot(t, T2, '-',  lw=0.8, label=r'$T_2$')
        ax.plot(t, E,  'k-', lw=1.0, alpha=0.6, label=r'$E_{\mathrm{полн}}$')

        ax.set_title(f"{method}, h = {dt}", fontsize=10)
        ax.set_xlabel("t, с")
        ax.set_ylabel("Энергия, Дж")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.show()

## 7. Сохранение энергии: относительное отклонение

In [ ]:
E0 = total_energy(state0[None, :])[0]  # начальная полная энергия

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
fig.suptitle("Относительное отклонение полной энергии от начальной", fontsize=12)

for ax, method in zip(axes, ["RK4", "Верле"]):
    for dt in dt_list:
        t, states = results[(method, dt)]
        E = total_energy(states)
        rel = (E - E0) / E0
        ax.plot(t, rel, lw=0.8, label=f"h = {dt}")

    ax.set_title(method)
    ax.set_xlabel("t, с")
    ax.set_ylabel(r"$(E - E_0)/E_0$")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(-3,3))

plt.show()

## 8. Сравнение численных частот с аналитическими

In [ ]:
# Длинное моделирование для точного определения частот через FFT
t_long = 10 * T_beat
dt_fft = 0.005
t_f, s_f = simulate(step_rk4, state0, dt_fft, t_long)
x1_f = s_f[:, 0]

N = len(t_f)
freqs = np.fft.rfftfreq(N, d=dt_fft)  # частоты в Гц
spectrum = np.abs(np.fft.rfft(x1_f))

# Ищем два главных пика
peak_indices = np.argsort(spectrum[1:])[-2:] + 1  # исключаем нулевую частоту
peak_freqs_hz = np.sort(freqs[peak_indices])
peak_freqs_rad = 2 * np.pi * peak_freqs_hz

print("Собственные частоты (рад/с):")
print(f"  omega_1: аналит. = {omega1:.4f},  числ. = {peak_freqs_rad[0]:.4f}")
print(f"  omega_2: аналит. = {omega2:.4f},  числ. = {peak_freqs_rad[1]:.4f}")
print()
print("Частота биений (рад/с):")
omega_beat_num = (peak_freqs_rad[1] - peak_freqs_rad[0]) / 2
print(f"  аналит. = {omega_beat:.4f},  числ. = {omega_beat_num:.4f}")
print(f"  Период биений: аналит. = {T_beat:.2f} с,  числ. = {2*np.pi/omega_beat_num:.2f} с")

# Спектр
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
omega_axis = 2*np.pi*freqs
ax.plot(omega_axis, spectrum / spectrum.max(), lw=0.8)
ax.axvline(omega1, color='red', ls='--', lw=1, label=f'$\\omega_1$ = {omega1:.2f}')
ax.axvline(omega2, color='green', ls='--', lw=1, label=f'$\\omega_2$ = {omega2:.2f}')
ax.set_xlim(0, 2*omega2)
ax.set_xlabel(r"$\omega$, рад/с")
ax.set_ylabel("Нормированная амплитуда")
ax.set_title(r"Спектр $x_1(t)$: два пика на собственных частотах")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 9. Сводная таблица ошибок

In [ ]:
print(f"{'Метод':<8} {'h':<8} {'max|ΔE|/E0':<16} {'max|x1 - x1_exact|':<20}")
print("-" * 56)

for method in ["RK4", "Верле"]:
    for dt in dt_list:
        t, states = results[(method, dt)]
        # Ошибка энергии
        E = total_energy(states)
        de = np.max(np.abs(E - E0)) / E0
        # Ошибка координат
        x1_ex, _, _, _ = exact_solution(t)
        dx = np.max(np.abs(states[:, 0] - x1_ex))
        print(f"{method:<8} {dt:<8.3f} {de:<16.3e} {dx:<20.3e}")